In [1]:
import coremltools as ct
from transformers import AutoTokenizer, LlamaForCausalLM
import numpy as np
from src.model import KvCacheStateLlamaForCausalLM
from src.converter import LlamaCoreMLConverter
import os

/Users/danielnewman/.virtualenvs/avi_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# print virtual environment
print(os.environ["VIRTUAL_ENV"])
# os.environ["TOKENIZERS_PARALLELISM"] = "false"

/Users/danielnewman/.virtualenvs/avi_venv


In [3]:


model_id = "meta-llama/Llama-3.2-1B-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

In [4]:
model = LlamaForCausalLM.from_pretrained(model_id)

model = KvCacheStateLlamaForCausalLM(
    model_id,
    batch_size=1,
    context_size=2048
)

converter = LlamaCoreMLConverter(
    model,
    batch_size=1,
    context_size=2048
)

mlmodel = converter.convert(quantize=True)
mlmodel.save("./models/llama-3.2-1b-instruct-quantized.mlpackage")
mlmodel


/Users/danielnewman/.virtualenvs/avi_venv/lib/python3.11/site-packages/transformers/modeling_utils.py:5006: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
Torch var valueCache is added again.
Torch var keyCache is added again.
Running MIL default pipeline:  69%|██████▉   | 61/88 [00:36<01:02,  2.30s/ passes]/Users/danielnewman/.virtualenvs/avi_venv/lib/python3.11/site-packages/coremltools/converters/mil/mil/passes/defs/optimize_repeat_ops.py:433: RuntimeWarning: overflow encountered in cast
  max(cur_range.low, tmp_range.low), min(cur_range.high, tmp_range.high)
Running compression pass linear_quantize_weights: 100%|██████████| 151/151 [00:20<00:00

input {
  name: "inputIds"
  type {
    multiArrayType {
      shape: 1
      shape: 1
      dataType: INT32
      shapeRange {
        sizeRanges {
          lowerBound: 1
          upperBound: 1
        }
        sizeRanges {
          lowerBound: 1
          upperBound: 2048
        }
      }
    }
  }
}
input {
  name: "causalMask"
  type {
    multiArrayType {
      shape: 1
      shape: 1
      shape: 1
      shape: 1
      dataType: FLOAT16
      shapeRange {
        sizeRanges {
          lowerBound: 1
          upperBound: 1
        }
        sizeRanges {
          lowerBound: 1
          upperBound: 1
        }
        sizeRanges {
          lowerBound: 1
          upperBound: 2048
        }
        sizeRanges {
          lowerBound: 1
          upperBound: 2048
        }
      }
    }
  }
}
output {
  name: "logits"
  type {
    multiArrayType {
      dataType: FLOAT16
    }
  }
}
state {
  name: "keyCache"
  type {
    stateType {
      arrayType {
        shape: 16
       

In [7]:

model = ct.models.MLModel('./models/llama-3.2-1b-instruct.mlpackage')

In [8]:
def _sample_token(logits, temperature):
    """
    Sample next token from logits using temperature
    """
    if temperature == 0:
        return int(np.argmax(logits))
        
    probs = np.exp(logits / temperature)
    probs = probs / np.sum(probs)
    return int(np.random.choice(len(probs), p=probs))

In [9]:
prompt = "Hello, how are you?"
inputs = tokenizer(prompt, return_tensors="np", padding=True)
input_ids = inputs["input_ids"].astype(np.int32)
current_input_ids = input_ids
generated_tokens = []

In [10]:
inputs = tokenizer(prompt, return_tensors="np", padding=True)
input_ids = inputs["input_ids"].astype(np.int32)
print(inputs.keys())
print(input_ids)
causal_mask = inputs["attention_mask"].astype(np.int32)
print(causal_mask)

dict_keys(['input_ids', 'attention_mask'])
[[128000   9906     11   1268    527    499     30]]
[[1 1 1 1 1 1 1]]


In [17]:
# Reshape to the specified shapes in the model spec
key_cache = np.zeros((1, 32, 2048, 128), dtype=np.float16)
value_cache = np.zeros((1, 32, 2048, 128), dtype=np.float16)


# Reshape inputs as before
current_input_ids = input_ids
causal_mask = np.ones((1, 1, len(current_input_ids[0]), len(current_input_ids[0])), dtype=np.float16)


state = model.make_state()

# Run inference with KV cache states

predictions = model.predict({
    'inputIds': current_input_ids.astype(np.int32),
    'causalMask': causal_mask.astype(np.float16)
}, state=state)


print(predictions)


{'logits': array([[[ 4.21875   ,  4.6796875 ,  7.8085938 , ..., -1.6855469 ,
         -1.6855469 , -1.6855469 ],
        [20.125     ,  8.765625  ,  6.2578125 , ..., -0.08374023,
         -0.08392334, -0.08416748],
        [ 9.4921875 ,  3.0996094 ,  2.6660156 , ...,  0.18481445,
          0.1842041 ,  0.18359375],
        ...,
        [11.328125  ,  7.8671875 ,  2.9902344 , ...,  1.0019531 ,
          1.0019531 ,  1.0009766 ],
        [15.5546875 ,  9.1484375 ,  4.7890625 , ...,  1.1083984 ,
          1.1074219 ,  1.1074219 ],
        [11.8515625 ,  8.3125    ,  6.7773438 , ..., -1.1962891 ,
         -1.1962891 , -1.1972656 ]]], dtype=float32)}


In [14]:
# take predictions and convert to tokens
logits = predictions['logits']
print(logits)

[[[ 4.21875     4.6796875   7.8085938  ... -1.6855469  -1.6855469
   -1.6855469 ]
  [20.125       8.765625    6.2578125  ... -0.08374023 -0.08392334
   -0.08416748]
  [ 9.4921875   3.0996094   2.6660156  ...  0.18481445  0.1842041
    0.18359375]
  ...
  [11.328125    7.8671875   2.9902344  ...  1.0019531   1.0019531
    1.0009766 ]
  [15.5546875   9.1484375   4.7890625  ...  1.1083984   1.1074219
    1.1074219 ]
  [11.8515625   8.3125      6.7773438  ... -1.1962891  -1.1962891
   -1.1972656 ]]]


In [15]:
# convert logits to tokens
tokens = np.argmax(logits, axis=-1)
print(tokens)


[[   2    0  358  649  499 3432  358]]


In [16]:
# convert tokens to text
text = tokenizer.decode(tokens[0], skip_special_tokens=True)
print(text)


#! I can you today I
